<center><h1>Satish_Manvika_HW3</h1></center>
<br>
<br>
Name: Manvika Satish
<br>
Github Username: manvsatish
<br>
USC ID: 5582061880

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import io
import re
import zipfile
import numpy as np
import pandas as pd

from pathlib import Path

Get the AReM Data Set

In [2]:
zip_path = Path("../data/AReM.zip")
print("zip file exists: ", zip_path.exists())
print("relative path: ", zip_path)
assert zip_path.exists(), f"Data file not found at {zip_path}"

zip file exists:  True
relative path:  ../data/AReM.zip


In [3]:
with zipfile.ZipFile(zip_path, "r") as archive:
    csv_files = sorted(name for name in archive.namelist() if name.endswith(".csv") and "__MACOSX" not in name)

print("Number of CSV files: ", len(csv_files))
print("first five files:")
for name in csv_files[:5]:
    print(name)

assert len(csv_files) == 88, f"Expected 88 csv files, found {len(csv_files)}"

Number of CSV files:  88
first five files:
AReM/bending1/dataset1.csv
AReM/bending1/dataset2.csv
AReM/bending1/dataset3.csv
AReM/bending1/dataset4.csv
AReM/bending1/dataset5.csv


In [4]:
column_names = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23",]
expected_times = np.arange(0, 120000, 250)


def read_activity_csv(archive, member_name):
    text = archive.read(member_name).decode("utf-8", errors="replace") #read and clean one arem csv file from zip archive
    rows = []

    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",") if "," in line else line.split() # most files use commas but bending2/dataset4 uses whitespace
        parts = [part.strip() for part in parts if part.strip()] # remove empty fields caused by trailing commas

        if len(parts) < 7:
            continue

        try:
            rows.append([float(value) for value in parts[:7]])
        except ValueError:
            continue #ignore any non-num header row

    data = pd.DataFrame(rows, columns=column_names)
    original_rows = len(data)
    data["time"] = data["time"].astype(int)
    data = data.groupby("time", as_index=False).mean()
    missing_times = sorted(set(expected_times) - set(data["time"]))

    needs_repair = (len(data) != len(expected_times) or not np.array_equal(data["time"].to_numpy(), expected_times))
    if needs_repair: # sitting/dataset8 is missing timestamp 13500
        data = (data.set_index("time").reindex(expected_times).interpolate(method="linear", limit_direction="both").rename_axis("time").reset_index())

    assert data.shape == (480, 7), (f"{member_name} has unexpected shape {data.shape}")
    assert not data.isna().any().any(), (f"{member_name} still contains missing values")
    cleaning_info = {"file": member_name, "original_rows": original_rows,"final_rows": len(data), "missing_times": missing_times, "repaired": needs_repair,}

    return data, cleaning_info

In [5]:
instances = []
cleaning_records = []

with zipfile.ZipFile(zip_path, "r") as archive:
    for member_name in csv_files:
        activity = Path(member_name).parent.name
        dataset_number = int(re.search(r"dataset(\d+)\.csv$", member_name).group(1))
        data, cleaning_info = read_activity_csv(archive, member_name)

        #for bending activities, datasets 1 and 2 are test data and for all other activities, datasets 1, 2, and 3 are test data
        if activity in {"bending1", "bending2"}:
            split = "test" if dataset_number in {1, 2} else "train"
        else:
            split = "test" if dataset_number in {1, 2, 3} else "train"

        instances.append({"activity": activity, "dataset": dataset_number, "split": split, "data": data,})
        cleaning_info.update({"activity": activity, "dataset": dataset_number,})
        cleaning_records.append(cleaning_info)

print("total instances loaded:", len(instances))
assert len(instances) == 88

total instances loaded: 88


In [6]:
cleaning_summary = pd.DataFrame(cleaning_records)
cleaning_summary.loc[cleaning_summary["repaired"], ["activity","dataset","original_rows","final_rows","missing_times",],]

,activity,dataset,original_rows,final_rows,missing_times
56,sitting,8,479,480,[13500]


#### Data loading and cleaning notes

The ZIP archive has 88 CSV files. I was able to spot some formatting inconsistencies (with the help of the Piazza post) and handled the issues during loading:
- `bending2/dataset4.csv` is whitespace-delimited, not comma-delimited.
- `cycling/dataset9.csv` has an extra trailing comma on the final data row.
- `cycling/dataset14.csv` has an extra trailing comma on the final data row.
- `sitting/dataset8.csv` has 479 observations and is missing timestamp 13500. The missing observation was restored by linear interpolation between its adjacent time points.

After cleaning, every instance has 480 observations and seven columns.

### (b) Test and Train Data

In [7]:
split_summary = (pd.DataFrame(instances).groupby(["activity", "split"]).size().unstack(fill_value=0).sort_index())
split_summary.loc["Total"] = split_summary.sum()
split_summary

split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12
Total,19,69


The first two datasets in `bending1` and `bending2` were assigned to the test set. For each of the other five activities, the first three datasets were assigned to the test set. All remaining datasets were assigned to the training set. This results in 19 test instances and 69 training instances.

### (c) Feature Extraction

#### i. Research

Common time-domain features used for time-series classification include minimum, maximum, mean, median, standard deviation, variance, range, first and third quartiles, interquartile range, selected percentiles, skewness, kurtosis, root mean square, and zero-crossing rate. These features summarize the central tendency, variability, distribution, magnitude, and shape of a time series.

#### ii. Extraction

In [8]:
sensor_columns = column_names[1:]

#extract the seven required features from each of the six time series
def extract_time_domain_features(data):
    features = {}
    for series_number, column in enumerate(sensor_columns, start=1):
        series = data[column]
        features[f"min{series_number}"] = series.min()
        features[f"max{series_number}"] = series.max()
        features[f"mean{series_number}"] = series.mean()
        features[f"median{series_number}"] = series.median()
        features[f"std{series_number}"] = series.std(ddof=1)
        features[f"1st_quart{series_number}"] = series.quantile(0.25)
        features[f"3rd_quart{series_number}"] = series.quantile(0.75)

    return features


activity_order = ["bending1","bending2","cycling","lying","sitting","standing","walking",]
ordered_instances = sorted(instances,key=lambda item: (activity_order.index(item["activity"]), item["dataset"],),)

feature_rows = []
for instance_number, instance in enumerate(ordered_instances, start=1):
    row = {"Instance": instance_number,"activity": instance["activity"],"dataset": instance["dataset"],"split": instance["split"],}
    row.update(extract_time_domain_features(instance["data"]))
    feature_rows.append(row)

feature_table = pd.DataFrame(feature_rows)
metadata_columns = ["Instance", "activity", "dataset", "split"]
feature_columns = [column for column in feature_table.columns if column not in metadata_columns]

assert feature_table.shape == (88, 46)
assert len(feature_columns) == 42
assert not feature_table[feature_columns].isna().any().any()

print("feature table shape:", feature_table.shape)
print("number of extracted time-domain features:", len(feature_columns))
feature_table.round(3)

feature table shape: (88, 46)
number of extracted time-domain features: 42


,Instance,activity,dataset,split,min1,max1,mean1,median1,std1,1st_quart1,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,1,bending1,1,test,37.25,45.00,40.625,40.50,1.477,39.25,...,2.188,33.000,36.00,0.00,1.92,0.571,0.43,0.583,0.000,1.300
1,2,bending1,2,test,38.00,45.67,42.813,42.50,1.436,42.00,...,1.995,32.000,34.50,0.00,3.11,0.571,0.43,0.601,0.000,1.300
2,3,bending1,3,train,35.00,47.40,43.954,44.33,1.559,43.00,...,2.000,35.362,36.50,0.00,1.79,0.493,0.43,0.514,0.000,0.940
3,4,bending1,4,train,33.00,47.75,42.180,43.50,3.671,39.15,...,3.849,30.458,36.33,0.00,2.18,0.614,0.50,0.524,0.000,1.000
4,5,bending1,5,train,33.00,45.75,41.678,41.75,2.243,41.33,...,2.411,28.458,31.25,0.00,1.79,0.383,0.43,0.389,0.000,0.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84,walking,11,train,19.50,45.33,33.587,34.25,4.651,30.25,...,3.284,13.730,18.25,0.00,8.32,3.260,3.11,1.640,2.050,4.322
84,85,walking,12,train,19.75,45.50,34.323,35.25,4.752,31.00,...,3.120,13.500,17.75,0.00,9.67,3.433,3.20,1.733,2.157,4.565
85,86,walking,13,train,19.50,46.00,34.546,35.25,4.842,31.25,...,2.823,14.000,17.75,0.00,10.00,3.338,3.08,1.657,2.160,4.335
86,87,walking,14,train,23.50,46.25,34.873,35.25,4.532,31.75,...,3.131,13.750,18.00,0.00,9.51,3.425,3.27,1.691,2.170,4.500


For each of the 88 instances, I extracted seven features from each of the six time series, producing 42 time-domain features. I used the features directly without normalization or standardization. The four additional columns identify the instance, activity, dataset number, and train/test split.

#### iii. Standard Deviation

In [9]:
feature_values = feature_table[feature_columns].to_numpy()
feature_standard_deviations = feature_values.std(axis=0, ddof=1) #sample standard deviation of each feature across all 88 instances
number_of_bootstrap_samples = 1000 # construct 90% percentile bootstrap confidence intervals
rng = np.random.default_rng(552)
bootstrap_standard_deviations = np.empty((number_of_bootstrap_samples, len(feature_columns)))

for bootstrap_sample in range(number_of_bootstrap_samples):
    sampled_indices = rng.integers(low=0, high=len(feature_values), size=len(feature_values),)
    bootstrap_sample_values = feature_values[sampled_indices]
    bootstrap_standard_deviations[bootstrap_sample] = (bootstrap_sample_values.std(axis=0, ddof=1))

confidence_interval_lower = np.percentile(bootstrap_standard_deviations, 5, axis=0,)
confidence_interval_upper = np.percentile(bootstrap_standard_deviations, 95, axis=0,)

standard_deviation_summary = pd.DataFrame({"Feature": feature_columns, "Estimated Standard Deviation": feature_standard_deviations, "90% CI Lower": confidence_interval_lower, "90% CI Upper": confidence_interval_upper,})

assert standard_deviation_summary.shape == (42, 4)
assert not standard_deviation_summary.isna().any().any()
assert (standard_deviation_summary["90% CI Lower"] <= standard_deviation_summary["90% CI Upper"]).all()

standard_deviation_summary.round(4)

,Feature,Estimated Standard Deviation,90% CI Lower,90% CI Upper
0,min1,9.5700,8.2353,10.6892
1,max1,4.3944,3.3326,5.3047
2,mean1,5.3357,4.7048,5.8863
3,median1,5.4401,4.7922,6.0093
4,std1,1.7722,1.5720,1.9438
5,1st_quart1,6.1536,5.5678,6.6341
6,3rd_quart1,5.1389,4.3299,5.8612
7,min2,0.0000,0.0000,0.0000
8,max2,5.0627,4.6400,5.3919
9,mean2,1.5742,1.3927,1.7044


I estimated each feature's sample standard deviation across all 88 instances using \(ddof=1\). I then generated 1,000 bootstrap samples by resampling the 88 instances with replacement and calculated a 90% percentile confidence interval using the 5th and 95th percentiles of the bootstrapped standard deviations.

#### iv. Select Features

I selected the minimum, mean, and maximum as the three most important time-domain features. The minimum and maximum describe the observed range and capture extreme activity levels, while the mean summarizes the overall level of the time series. Together, these features give me a compact description of location and range and are straightforward to interpret across activities.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regression will have a training RSS that is less than or equal to that of the linear regression. The linear model is nested within the cubic model because the cubic model becomes linear when the coefficients of \(X^2\) and \(X^3\) are set to zero. Therefore, least squares can always choose those coefficients to reproduce the linear fit, while the additional terms may reduce the training RSS further. Even though the true relationship is linear, the cubic model may fit some random noise in the training data.

### (b) Linear Test

The linear regression is expected to have the lower test RSS because the true relationship is linear. The cubic model introduces unnecessary \(X^2\) and \(X^3\) terms, increasing model variance without reducing the bias. These additional terms could fit noise in the training data and therefore generalize less effectively to observations that are unseen. This is an expectation rather than a guarantee for every particular test set.

### (c) Not Linear Train

The cubic regression will have a training RSS that is less than or equal to that of the linear regression. This follows from the nesting of the models. The cubic model can reproduce the linear model by setting the coefficients of \(X^2\) and \(X^3\) to zero. Its additional flexibility can only leave the RSS unchanged or reduce it, regardless of the unknown form or degree of the true nonlinearity.

### (d) Not Linear Testing

There is not enough information to determine which model will have the lower test RSS. If the true relationship is close enough to a cubic function, the cubic model may reduce bias and get a lower test RSS. If the true relationship is only mildly nonlinear, has a form that is not well approximated by a cubic polynomial, or the noise level is high, the cubic model's additional variance could lead to higher test RSS. Therefore the result depends on the unknown relationship and the bias-variance tradeoff.